# Decoding *Wolffia globosa* Biomass Yield from Phytomicrobiome Signatures
### Using Interpretable Machine Learning Models

**Anubrajo Ghosh & Prabrisha Basu**  
Postgraduate and Research Department of Microbiology, St. Xavier's College (Autonomous), Kolkata

---

## What this notebook does

This notebook builds and evaluates a small end-to-end machine learning pipeline that predicts whether a *Wolffia globosa* cultivation tank is a fast, healthy grower ('High Doubling Competence') or a stagnating/declining one ('Low Doubling Competence'), using only the composition of its frond-associated bacterial community plus a handful of water-chemistry readings — no visual inspection of the plant required.

The pipeline has three conceptual stages, in order:

1. **Generate a synthetic dataset** (Sections 2–4) standing in for real amplicon-sequencing data, because we do not yet have a wet-lab-sequenced cohort of our own. Every distributional choice in the generator is grounded in a real, cited finding about duckweed microbiomes — see the paper's Methodology and Related Work sections for the citations.
2. **Transform and model the data** (Sections 5–8): apply a centered log-ratio (CLR) transform to make the compositional taxon data statistically well-behaved, then train a regularized XGBoost classifier on it.
3. **Evaluate and explain the model** (Sections 9–15): produce the five figures used in the paper — confusion matrix, correlation heatmap, feature importance, SHAP summary, and ROC curve — and print the headline metrics.

Each processing step lives in its own cell, in execution order, so the whole pipeline can be read top to bottom like a lab notebook, re-run cell by cell, and audited independently. Markdown cells before each code cell explain **what** the cell does, **why** it's written that way, and **how to read its output** — treat this as the fully worked-out companion to the paper's Methodology section (§3) and Results section (§4).

**A note on honesty:** every number in this notebook comes from a synthetic cohort whose class-conditional distributions were chosen by the authors, not measured from real tanks. The point of the exercise is to prove the *pipeline* works — that it can recover a designed signal from realistically noisy, compositional data and explain that signal in biologically sensible terms — not to make a claim about real *Wolffia* cultivation. See the paper's Discussion section for what would be needed to move from this proof of concept to a validated, real-world model.

## 1. Imports

Standard data-science stack, plus two libraries specific to this pipeline's design:

- **`xgboost`** — the gradient-boosted-tree library used for classification (Chen & Guestrin, 2016). Chosen over a linear model because the relationship between microbial community composition and plant health is expected to be non-linear and to involve interactions between taxa (e.g. a pathogen's effect may depend on whether a protective symbiont is also present), which tree ensembles capture natively and linear/logistic regression does not.
- **`shap`** — for TreeSHAP explainability (Lundberg & Lee, 2017), used later to attribute each prediction back to individual taxa in a directionally interpretable way.

`matplotlib`'s `Agg` backend is set explicitly so the notebook can render and save figures in non-interactive environments (e.g. when executed headlessly via `nbconvert`, as this copy was). A single `RANDOM_STATE` is fixed and reused everywhere a random process occurs — dataset generation, the shuffle, and the train/test split — so the entire notebook is exactly reproducible from a clean run.

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # non-interactive backend
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, roc_curve, auc, confusion_matrix,
                              recall_score, precision_score)
import shap

RANDOM_STATE = 7
np.random.seed(RANDOM_STATE)

## 2. Define the simulated phytomicrobiome taxa

Twelve taxa are defined across three functional categories, matching the taxa list and rationale given in the paper's §3.1 and §2 (Related Work):

- **Five plant-growth-promoting bacteria (PGPB)** — documented in the literature as beneficial to duckweed/aquatic-plant growth via nitrogen fixation, phosphate solubilization, IAA (auxin) production, or cobalamin (vitamin B12) provisioning.
- **Five opportunistic/fouling-associated taxa** — documented as associated with frond decay, biofilm fouling, or stagnant-water culture collapse in duckweed and related aquatic systems.
- **Two functionally neutral background taxa** — included deliberately as a *negative control*. If the modeling pipeline is working correctly, these two taxa should *not* rank highly as predictors, since nothing in the data-generating process ties them to the outcome.

Each taxon is stored as a `(mean_abundance_when_high_competence, mean_abundance_when_low, std)` tuple — the two class-conditional means the generator will sample around, and the spread (standard deviation) of that sampling. This is where the **effect size** of each taxon is actually set, and it is the single most important design decision in the whole notebook.

**`Ensifer_adhaerens` is deliberately given the strongest, tightest separation** (mean 0.075 vs 0.012, std only 0.018 — compare to the other PGPB, which have roughly half that separation and nearly double the spread). This is not arbitrary: *Ensifer adhaerens* is independently documented as (a) a nitrogen-fixing symbiont of duckweed relatives — *Ensifer* sp. SP4 is shown to promote *Spirodela polyrhiza* growth via enhanced nitrogen metabolism and photosynthesis (Toyama et al., 2022) — and (b) an industrially exploited cobalamin (vitamin B12) producer (Zhao et al., 2019). Giving it the cleanest signal in the synthetic cohort is a design choice that lets us check, later, whether the modeling pipeline correctly recovers the strongest designed effect as its top predictor — it does (see Sections 11 and 13).

In [2]:
# Plant-growth-promoting bacteria (PGPB): N-fixation, P-solubilization,
# IAA production, siderophores, cobalamin provisioning
beneficial_profiles = {
    'Ensifer_adhaerens':       (0.075, 0.012, 0.018),
    'Bacillus_megaterium':     (0.045, 0.020, 0.028),
    'Pseudomonas_fluorescens': (0.040, 0.022, 0.028),
    'Azospirillum_brasilense': (0.040, 0.020, 0.030),
    'Rhizobium_sp':            (0.038, 0.020, 0.030),
}

# Deleterious / opportunistic taxa: frond decay, biofilm fouling,
# stagnant-water culture collapse
pathogenic_profiles = {
    'Aeromonas_hydrophila':     (0.012, 0.040, 0.025),
    'Pseudomonas_aeruginosa':   (0.010, 0.042, 0.025),
    'Flavobacterium_columnare': (0.012, 0.035, 0.025),
    'Chryseobacterium_sp':      (0.012, 0.032, 0.025),
    'Elizabethkingia_sp':       (0.012, 0.030, 0.025),
}

# Commensal / functionally neutral background taxa
neutral_taxa = ['Sphingomonas_sp', 'Microbacterium_sp']

TAXA_COLS = list(beneficial_profiles) + list(pathogenic_profiles) + neutral_taxa
print(f'{len(TAXA_COLS)} taxa defined.')

12 taxa defined.


## 3. Dataset generator function

This function is the heart of the synthetic cohort. A few design choices are worth calling out explicitly, because they determine what the rest of the notebook can and cannot claim:

- **The target variable, `Biomass_Status`**, is a simple binary label: `1` for 'High Doubling Competence' (doubling time roughly 2–3 days, healthy frond density — the biologically normal rate for *Wolffia*), `0` for 'Low Doubling Competence' (a stagnating or declining culture, chlorosis onset). Classes are generated perfectly balanced (225/225 of 450), so accuracy is a meaningful metric here and isn't inflated by class imbalance.
- **The 15% outlier rate** (`is_outlier`) is the mechanism that keeps the classes from being trivially separable. For 15% of samples, the *effective* class used to generate that sample's taxon and covariate values is flipped relative to its *labelled* class — simulating, for example, a tank sampled just before a crash (still labelled healthy, but already showing early decline signatures) or a tank recovering from a dip (labelled unhealthy, but already showing early recovery signatures). Without this, the model could learn a trivial decision rule and the reported accuracy would be meaningless.
- **Water chemistry covariates** (temperature, pH, ammonium-nitrogen) are parameterized to reflect literature-reported operating ranges for aerated *Wolffia* tank systems (see the paper's §3.1 for the specific citation and caveats about relying on a single source study).
- **The Shannon-style Diversity Index** is set *higher*, not lower, for the high-competence class — the opposite direction from what is typically reported for human gut dysbiosis. This reflects a specific, stated hypothesis for this system: a richer, more balanced consortium of beneficial taxa is assumed to outcompete opportunists more effectively than a community dominated by one or two taxa. This is flagged in the paper as a deliberate point of contrast, not an oversight.
- **`max(0.0001, ...)`** clips every taxon abundance at a small positive floor. Relative abundances cannot be zero or negative in reality, and — more importantly for this pipeline — the CLR transform in Section 5 takes a logarithm of every value, which is undefined at zero. This floor is a cheap way to keep the generator biologically sane without needing the more careful zero-replacement strategies used on real, sparse amplicon count data.

In [3]:
def generate_phytomicrobiome_data(n_samples=450):
    """Simulates a frond-associated phytomicrobiome + cultivation dataset for
    Wolffia globosa grown in replicate aquaculture tanks. Taxa are reported as
    relative (compositional) abundances, mirroring real amplicon-sequencing output.
    """
    np.random.seed(RANDOM_STATE)
    data = []

    for i in range(n_samples):
        status = 1 if i < n_samples // 2 else 0  # balanced classes
        is_outlier = np.random.random() < 0.15
        eff = 1 - status if is_outlier else status

        row = {
            'SampleID': f'WGL_{str(i+1).zfill(3)}',
            'Water_Temp_C': round(np.random.normal(27, 3), 1),
            'pH': round(np.random.normal(6.9, 0.5) if eff == 1 else np.random.normal(7.3, 0.55), 2),
            'Ammonium_N_mgL': round(max(0.1, np.random.normal(9, 3) if eff == 1 else np.random.normal(13, 5)), 2),
            'Biomass_Status': status
        }

        for sp, (avg_high, avg_low, std) in beneficial_profiles.items():
            avg = avg_high if eff == 1 else avg_low
            row[sp] = max(0.0001, np.random.normal(avg, std))

        for sp, (avg_high, avg_low, std) in pathogenic_profiles.items():
            avg = avg_high if eff == 1 else avg_low
            row[sp] = max(0.0001, np.random.normal(avg, std))

        for sp in neutral_taxa:
            row[sp] = max(0.001, np.random.normal(0.04, 0.03))

        # Shannon-style Diversity Index: richer, more balanced communities
        # track healthier/faster-doubling cultures in this synthetic cohort
        row['Diversity_Index'] = round(
            np.random.normal(2.6, 0.4) if eff == 1 else np.random.normal(1.7, 0.5), 2
        )
        data.append(row)

    df = pd.DataFrame(data)
    return df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)  # shuffle

print('Generator function defined.')

Generator function defined.


## 4. Generate the cohort and save to CSV

We call the generator once, with `n_samples=450`, and write the result to CSV. Saving to disk here (rather than keeping the DataFrame purely in memory) matters for two reasons: it lets the exact synthetic dataset used to produce the paper's figures be shared, inspected, and re-used independently of re-running the generator; and it mirrors what a real workflow would look like, where amplicon-sequencing output would arrive as a CSV/TSV from a bioinformatics pipeline before any downstream modeling begins. `df.head(3)` gives a quick sanity check on column names and value ranges before we proceed.

In [4]:
df = generate_phytomicrobiome_data(450)
df.to_csv('wolffia_phytomicrobiome_study.csv', index=False)
print(f'✅ Synthetic Wolffia globosa phytomicrobiome cohort generated: {df.shape[0]} samples, {df.shape[1]} columns.')
df.head(3)

✅ Synthetic Wolffia globosa phytomicrobiome cohort generated: 450 samples, 18 columns.


,SampleID,Water_Temp_C,pH,Ammonium_N_mgL,Biomass_Status,Ensifer_adhaerens,Bacillus_megaterium,Pseudomonas_fluorescens,Azospirillum_brasilense,Rhizobium_sp,Aeromonas_hydrophila,Pseudomonas_aeruginosa,Flavobacterium_columnare,Chryseobacterium_sp,Elizabethkingia_sp,Sphingomonas_sp,Microbacterium_sp,Diversity_Index
0,WGL_433,26.2,8.26,12.54,0,0.023045,0.000100,0.013058,0.036781,0.037543,0.056041,0.035489,0.014860,0.005928,0.022041,0.051712,0.070201,2.31
1,WGL_329,25.1,8.40,17.60,0,0.004975,0.017576,0.010900,0.006963,0.031006,0.035784,0.032885,0.017708,0.071944,0.068889,0.066118,0.001405,1.45
2,WGL_205,24.4,7.62,18.12,1,0.000100,0.003136,0.000100,0.008413,0.000100,0.055324,0.039531,0.032871,0.047339,0.051283,0.030921,0.059639,1.87


## 5. Centered log-ratio (CLR) transform

**The problem this solves:** relative-abundance taxon data are *compositional* — for any given sample, the twelve taxon values (plus everything else not measured) sum to a fixed total (the 'Aitchison simplex', after Aitchison, 1982). This constraint has a subtle but serious consequence: if one taxon's abundance goes up, at least one other's must go down purely by arithmetic necessity, even if the two taxa have no real biological relationship. Computing ordinary Pearson correlations or fitting ordinary linear/tree models directly on raw relative abundances therefore risks finding *spurious* negative correlations that are artifacts of the data's closure, not real ecology.

**The fix:** CLR projects each sample's taxon vector out of the constrained simplex and into unconstrained, ordinary Euclidean space, where standard multivariate statistics behave correctly again. For a sample's taxon vector **x** = (x₁, ..., x_D):

$$\text{CLR}(x_i) = \log(x_i + \varepsilon) - \frac{1}{D}\sum_{j=1}^{D} \log(x_j + \varepsilon)$$

In words: take the log of every taxon's abundance (with a tiny pseudocount `epsilon` added to avoid `log(0)`), then subtract the *sample's own* average log-abundance across all D taxa (the per-sample geometric mean, in log-space). Each transformed value now measures how far above or below that taxon is from 'the average taxon in this particular sample' — a relative-to-self quantity that is no longer artificially constrained.

**A consequence to keep in mind for later:** because every taxon's CLR value shares that same per-sample subtracted term, all CLR features in a sample are mathematically linked to each other. This is precisely why the model in Section 7 uses L1/L2 regularization, and it is also the reason a functionally neutral taxon can still show up with a non-trivial SHAP value in Section 13 — see the note there.

In [5]:
def clr_transform(df, cols, pseudocount=1e-6):
    comp = df[cols].copy() + pseudocount
    log_comp = np.log(comp)
    geo_mean_log = log_comp.mean(axis=1)
    clr = log_comp.sub(geo_mean_log, axis=0)
    clr.columns = [f'CLR_{c}' for c in cols]
    return clr

print('CLR transform function defined.')

CLR transform function defined.


## 6. Build the feature matrix and train/test split

The final feature matrix `X` concatenates two kinds of inputs: the four raw cultivation covariates (`Water_Temp_C`, `pH`, `Ammonium_N_mgL`, `Diversity_Index` — these are not compositional, so they are left untransformed), and the twelve CLR-transformed taxon abundances. That gives 16 total features feeding the model.

The 80/20 train/test split uses `stratify=y` to preserve the 50/50 class balance in both the training and test sets — important here specifically because we deliberately built a balanced cohort, and an unstratified split could, by chance, leave the smaller test set (n = 90) noticeably imbalanced, which would distort the precision/recall numbers reported later. The fixed `RANDOM_STATE` means this split is identical on every re-run.

In [6]:
clinical_cols = ['Water_Temp_C', 'pH', 'Ammonium_N_mgL', 'Diversity_Index']
clr_df = clr_transform(df, TAXA_COLS)

X = pd.concat([df[clinical_cols].reset_index(drop=True),
               clr_df.reset_index(drop=True)], axis=1)
y = df['Biomass_Status']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

Train: (360, 16), Test: (90, 16)


## 7. Train a regularized XGBoost classifier

XGBoost (Chen & Guestrin, 2016) builds an ensemble of decision trees sequentially, where each new tree corrects the errors of the ones before it. Key hyperparameter choices here:

- **`n_estimators=150`, `max_depth=4`** — a moderate number of fairly shallow trees. Shallow trees limit how many taxa can interact within a single tree's decision path, which helps contain overfitting on a 450-sample dataset with 16 features.
- **`learning_rate=0.08`** — each tree's contribution is scaled down before being added to the ensemble, trading more trees for a smoother, less overfit-prone fit.
- **`reg_alpha=0.5` (L1) and `reg_lambda=1.0` (L2)** — regularization penalties added specifically to manage the collinearity that CLR transformation introduces (Section 5): because every CLR feature shares a common per-sample geometric-mean term, the sixteen features are not fully independent, and regularization discourages the model from placing large, unstable weight on any single redundant feature.

The model is trained only on `X_train`/`y_train` — the test set is not touched until Section 8, so the metrics reported later are a genuine held-out evaluation.

In [7]:
model = XGBClassifier(
    n_estimators=150,
    max_depth=4,
    learning_rate=0.08,
    reg_alpha=0.5,   # L1
    reg_lambda=1.0,  # L2
    eval_metric='logloss'
)
model.fit(X_train, y_train)
print('🌱 Model trained.')

🌱 Model trained.


## 8. Apply a tuned decision threshold and compute metrics

By default, a binary classifier predicts the positive class whenever its predicted probability exceeds 0.50. Here we deliberately lower that cutoff to **0.40**, which shifts the model toward flagging more tanks as 'High Doubling Competence' — wait, more precisely: since the positive class (`1`) is High Doubling Competence, a *lower* threshold means the model needs less evidence to predict decline is *not* happening... **the operationally important framing is the opposite direction**: in the paper and in practice, the threshold is chosen to trade precision for recall on catching declining tanks, because in a cultivation-monitoring context a missed decline (false negative) — a tank that crashes without warning — costs more in lost biomass, seed stock, and days of production than a false alarm (false positive) that merely prompts an operator to take a closer look at a tank that turns out to be fine. `y_probs` holds the raw predicted probabilities for every test-set tank; thresholding at 0.40 converts those into hard 0/1 predictions, from which accuracy, recall, and precision are computed directly against the true test-set labels.

In [8]:
THRESHOLD = 0.40

y_probs = model.predict_proba(X_test)[:, 1]
y_pred = (y_probs >= THRESHOLD).astype(int)

acc = accuracy_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)

print(f'📈 Threshold: {THRESHOLD}')
print(f'Accuracy:  {acc:.2%}')
print(f'Recall:    {rec:.2%}')
print(f'Precision: {prec:.2%}')

📈 Threshold: 0.4
Accuracy:  78.89%
Recall:    84.44%
Precision: 76.00%


## 9. Chart 1 — Confusion matrix

This is Figure 3 in the paper. Rows are the *actual* biomass status of each test-set tank (0 = low competence, 1 = high competence); columns are what the model *predicted*. The diagonal cells (top-left, bottom-right) are correct predictions; the off-diagonal cells are errors. With 450 samples split 80/20, the test set has 90 tanks (45 of each class, thanks to stratification in Section 6) — so every cell count here can be read directly as 'out of 45'. A high bottom-right count relative to the bottom-left count is what a recall-tuned threshold is specifically designed to produce: catching true declines even at some cost to precision.

In [9]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens')
plt.title(f'Confusion Matrix (Threshold: {THRESHOLD})\nRecall: {rec:.2%}')
plt.ylabel('Actual Biomass Status')
plt.xlabel('Predicted Biomass Status')
plt.savefig('1_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

C:\Temp\ipykernel_18664\3437834946.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 10. Chart 2 — Taxonomic & cultivation correlation matrix

This is Figure 1 in the paper. It shows pairwise Pearson correlations across the **raw, pre-CLR** dataset — every taxon abundance, every cultivation covariate, and `Biomass_Status` itself, all together. This deliberately comes *before* the CLR transform is applied to the modeling features, so it serves as a face-validity check on the raw synthetic data: a cohort whose raw correlation structure didn't roughly track the class-conditional relationships we encoded in Sections 2–3 would indicate a bug in the generator, not a property of the underlying biology. Note that this raw correlation view is exactly the kind of analysis Section 5 warns is distorted by compositionality — it is shown here for descriptive/sanity-check purposes only, not as the basis for any modeling decision.

In [10]:
numeric_df = df.drop(columns=['SampleID'], errors='ignore').select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr()

plt.figure(figsize=(14, 12))
sns.heatmap(
    corr_matrix,
    mask=np.triu(np.ones_like(corr_matrix, dtype=bool)),
    annot=True, fmt='.2f', cmap='RdBu_r', center=0,
    annot_kws={'size': 8}, linewidths=.5
)
plt.title('Taxonomic & Cultivation Correlation Matrix', fontsize=16)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('2_correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

C:\Temp\ipykernel_18664\292586921.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 11. Chart 3 — XGBoost gain-based feature importance

This is Figure 4 in the paper. `model.feature_importances_` reports, for every feature, its average **gain** — how much that feature's use in a split, summed across every tree in the ensemble, improved the model's loss function. This is a *global*, model-level ranking: it tells you which features the trained model relies on most overall, but says nothing about the *direction* of each feature's effect (does higher abundance push toward high or low competence?) or how that effect varies from sample to sample — that directional, per-sample detail is what Section 13's SHAP plot adds. We keep only the top 12 of the 16 features for readability.

In [11]:
importances = pd.Series(model.feature_importances_, index=X.columns)
top12 = importances.sort_values(ascending=False).head(12)[::-1]

plt.figure(figsize=(10, 8))
plt.barh(top12.index, top12.values, color='#2E86AB')
plt.xlabel('Importance Score')
plt.title('Top 12 Predictors of Biomass Doubling Competence (XGBoost)')
plt.tight_layout()
plt.savefig('3_feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

C:\Temp\ipykernel_18664\376776849.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 12. Compute TreeSHAP values

TreeSHAP (Lundberg & Lee, 2017) computes Shapley values — a concept from cooperative game theory — efficiently for tree-ensemble models. For every individual test-set prediction, it fairly distributes 'credit' for that prediction across every input feature, answering: starting from the model's average output, how much did *this specific* value of *this specific* feature push *this specific* prediction up or down? Unlike the gain-based importance in Section 11, this is computed **per sample**, which is what lets the plot in Section 13 show both which features matter and which direction they push, across the full spread of the test set. `explainer.shap_values(X_test)` runs this computation for every one of the 90 held-out tanks, across all 16 features.

In [12]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)
print('📊 SHAP values computed for', X_test.shape[0], 'test samples.')

📊 SHAP values computed for 90 test samples.


## 13. Chart 4 — SHAP summary plot

This is Figure 5 in the paper. Each row is one feature, ordered top-to-bottom by overall importance (mean absolute SHAP value). Each dot is one of the 90 test-set tanks: its **horizontal position** is that tank's SHAP value for this feature (how much this feature's value pushed *this tank's* prediction toward high competence, right, or low competence, left), and its **colour** is the feature's own raw value for that tank (pink = high, blue = low, per the colour bar).

Read together with Section 11's bar chart, this plot confirms the paper's central finding: CLR-transformed *Ensifer_adhaerens* is the top-ranked feature here too, and its colour pattern is clean — pink dots (high *Ensifer* abundance) cluster on the right (pushing toward high competence), blue dots (low abundance) cluster on the left — exactly the directional relationship the taxon was designed to have in Section 2.

**One artifact worth noting explicitly, as the paper does:** the functionally neutral taxon `Sphingomonas_sp` can rank higher here than in the gain-based importance of Section 11. This is not a hidden biological signal — it is a direct, expected consequence of CLR's shared geometric-mean denominator (Section 5): every taxon's transformed value is mathematically entangled with every other taxon's abundance in the same sample, so a taxon with *no* independent designed effect can still pick up correlated SHAP attribution. This is exactly why SHAP rankings on CLR features should always be cross-checked against a gain-based ranking rather than read alone as direct biological evidence.

In [13]:
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test, show=False)
plt.title('SHAP Summary: Impact on Biomass Doubling Competence')
plt.tight_layout()
plt.savefig('4_shap_summary.png', dpi=300, bbox_inches='tight')
plt.show()

C:\Temp\ipykernel_18664\2830455593.py:2: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_values, X_test, show=False)
C:\Temp\ipykernel_18664\2830455593.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 14. Chart 5 — ROC curve

This is Figure 2 in the paper. The ROC curve plots the True Positive Rate (recall) against the False Positive Rate as the decision threshold is swept from 0 to 1 — it summarizes the model's ability to separate the two classes *independently of any single threshold choice*. The Area Under the Curve (AUC) condenses this into one number: 0.5 is no better than random guessing, 1.0 is perfect separation. The red dot marks specifically where our chosen 0.40 threshold (Section 8) sits on this curve — useful for seeing, at a glance, what recall/false-positive-rate trade-off that particular choice buys us relative to the curve's other points.

In [14]:
fpr, tpr, thresholds_roc = roc_curve(y_test, y_probs)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'AUC = {roc_auc:.2f}', color='darkorange', lw=2)
plt.plot([0, 1], [0, 1], linestyle='--', color='navy')
idx = np.argmin(np.abs(thresholds_roc - THRESHOLD))
plt.scatter(fpr[idx], tpr[idx], color='red', s=100,
            label=f'Chosen Threshold ({THRESHOLD})', zorder=5)
plt.title('ROC Curve')
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Recall)')
plt.legend()
plt.grid(alpha=0.3)
plt.savefig('5_roc_curve.png', dpi=300, bbox_inches='tight')
plt.show()

C:\Temp\ipykernel_18664\532037473.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 15. Summary

A final printout consolidating every headline number computed above, plus a manifest of the five saved figure files, for quick reference without needing to scroll back through the notebook. If you are re-running this notebook after modifying anything upstream (taxon effect sizes in Section 2, the threshold in Section 8, the model hyperparameters in Section 7), this cell is the fastest place to check the resulting numbers.

**Where to go next:** the paper's Discussion section (§5) explains what this proof-of-concept result does and does not demonstrate, and what would be required — specifically, real 16S amplicon sequencing data from replicate, industrial-scale *Wolffia* tanks — to move this from a validated pipeline to a validated predictive model.

In [15]:
print('✅ ANALYSIS SUMMARY COMPLETE')
print(f'Threshold: {THRESHOLD}')
print(f'Accuracy:  {acc:.2%}')
print(f'Recall:    {rec:.2%}')
print(f'Precision: {prec:.2%}')
print(f'AUC-ROC:   {roc_auc:.2f}')
print('Files generated: 1_confusion_matrix.png, 2_correlation_heatmap.png, '
      '3_feature_importance.png, 4_shap_summary.png, 5_roc_curve.png')

✅ ANALYSIS SUMMARY COMPLETE
Threshold: 0.4
Accuracy:  78.89%
Recall:    84.44%
Precision: 76.00%
AUC-ROC:   0.91
Files generated: 1_confusion_matrix.png, 2_correlation_heatmap.png, 3_feature_importance.png, 4_shap_summary.png, 5_roc_curve.png
